# SEED-VII EEG-Conformer 端到端流水线

本 Notebook 一键依次调用四个独立 `.py` 脚本：

1. `scripts/merge_volumes.py` — 32 分卷 zip 流式校验（默认**不落盘**完整 zip）
2. `scripts/preprocess_to_npz.py` — 流式预处理 → 单个 `.npz`（含 trial-level 划分）
3. `scripts/train.py` — 训练 / 续训（断点续训 + 10h 软超时）
4. `scripts/encode.py` — 用训练好的编码器导出 embedding
5. `scripts/upload_to_modelscope.py` — 上传到 `DEREKVERSE/SEED-VII`

> 严格遵循 `Design.md` 的所有原则。

## 0. 环境与路径

In [ ]:
import os, sys, subprocess, json, pathlib
REPO = pathlib.Path.cwd().resolve()
if REPO.name == 'notebooks':
    REPO = REPO.parent
print('REPO =', REPO)

# 修改为你的实际路径
VOLUMES_DIR  = '/data/seed_vii_volumes'             # 32 个分卷所在目录
VOL_PATTERN  = '*.zip.*'                            # 形如 SEED-VII.zip.001
SAVE_INFO    = '/data/seed_vii_save_info'           # save_info 文件夹（连续标签 CSV）
WORK         = pathlib.Path('/workspace/seed_vii'); WORK.mkdir(parents=True, exist_ok=True)
NPZ_OUT      = str(WORK / 'preprocessed' / 'seed_vii.npz')
RUN_DIR      = str(WORK / 'runs_seed_vii')
ENC_OUT      = str(WORK / 'encoded' / 'embeddings.npz')

MS_DATASET   = 'DEREKVERSE/SEED-VII'
MS_TOKEN     = os.environ.get('MODELSCOPE_API_TOKEN', '')  # 也可以在下面 cell 里直接赋值

def run(cmd, env=None):
    print('$', ' '.join(cmd))
    res = subprocess.run(cmd, cwd=str(REPO), env={**os.environ, **(env or {})})
    if res.returncode != 0:
        raise SystemExit(f'cmd failed with code {res.returncode}')
PY = sys.executable
print('python =', PY)

In [ ]:
# 一次性安装依赖（如有需要）
# !pip install -q -r requirements.txt

## 1. 校验 32 个分卷（不落盘大 zip）

In [ ]:
run([PY, 'scripts/merge_volumes.py',
     '--volumes-dir', VOLUMES_DIR,
     '--pattern', VOL_PATTERN])
# 如确有 >200G 临时盘，可追加 --concat-to /scratch/SEED-VII.zip 物理合并；
# 默认推荐：保持流式，下一步直接虚拟拼接读取。

## 2. 流式预处理 → 单个 npz

**先切分（trial-level），再做窗口/标准化** → 杜绝数据泄漏。
每个 clip 居中 60% + 4 秒窗口 + 50% 重叠 + 按通道 z-score。
连续强度从 `save_info` 解析。

In [ ]:
run([PY, 'scripts/preprocess_to_npz.py',
     '--volumes-dir', VOLUMES_DIR,
     '--pattern', VOL_PATTERN,
     '--save-info-dir', SAVE_INFO,
     '--output', NPZ_OUT,
     '--val-ratio', '0.1',
     '--test-ratio', '0.1',
     '--split-unit', 'trial',
     '--max-windows-per-trial', '60'])

## 3. 训练（先分类预训练，再联合训练；10h 自动安全保存）

退化方案默认 **不开** ranking 损失（`--enable-rank` 关），只用分类 + 强度回归。
如需开启：加 `--enable-rank` 并可调 `--gamma-rank-end`。

In [ ]:
run([PY, 'scripts/train.py',
     '--data', NPZ_OUT,
     '--output-dir', RUN_DIR,
     '--device', 'auto', '--amp',
     '--pretrain-epochs', '10',
     '--max-epochs', '200',
     '--batch-size', '64',
     '--lr', '2e-4', '--min-lr', '1e-5',
     '--sample-weight-mode', 'continuous',
     '--max-runtime-hours', '10'])

In [ ]:
# 续训：进程中断后随时调这一行
# run([PY, 'scripts/train.py', '--data', NPZ_OUT, '--output-dir', RUN_DIR,
#      '--device', 'auto', '--amp', '--resume', '--max-runtime-hours', '10'])

## 4. 编码导出

In [ ]:
run([PY, 'scripts/encode.py',
     '--data', NPZ_OUT,
     '--checkpoint', str(pathlib.Path(RUN_DIR) / 'best_encoder.pt'),
     '--output', ENC_OUT,
     '--feature-type', 'projected',
     '--device', 'auto', '--amp'])
import numpy as np; z = np.load(ENC_OUT, allow_pickle=True)
print({k: getattr(z[k], 'shape', z[k]) for k in z.files})

## 5. 上传合并后的 zip 到 ModelScope

若你确实物理拼接了 160GB zip（步骤 1 用 `--concat-to`），
下面会把它整个 push 回 `DEREKVERSE/SEED-VII`。
也可改成 `--local-dir` 上传整个分卷目录。

In [ ]:
# os.environ['MODELSCOPE_API_TOKEN'] = 'paste-your-token-here'
LOCAL_ZIP = '/scratch/SEED-VII.zip'  # 仅当步骤 1 物理合并时存在
if pathlib.Path(LOCAL_ZIP).exists():
    run([PY, 'scripts/upload_to_modelscope.py',
         '--local-file', LOCAL_ZIP,
         '--dataset', MS_DATASET,
         '--path-in-repo', 'data/SEED-VII.zip'])
else:
    print('No merged zip found. Alternative: upload the volumes folder.')
    # run([PY, 'scripts/upload_to_modelscope.py',
    #      '--local-dir', VOLUMES_DIR,
    #      '--dataset', MS_DATASET,
    #      '--path-in-repo', 'volumes',
    #      '--allow', '*.zip.*'])